# MiniMind — Serious Pre-training + SFT on Google Colab

This notebook reproduces the **full-size** MiniMind training pipeline from the
[upstream repo](https://github.com/jingyaogong/minimind) inside Colab.

Compared to `minimind_data_pretrain.ipynb` (which uses a `MockTokenizer` and the
`*_mini` datasets purely as a teaching demo), this notebook:

| Aspect | Mini demo notebook | This serious notebook |
|---|---|---|
| Tokenizer | `MockTokenizer` (char → id) | **Real BPE tokenizer** from `model/tokenizer.json` (vocab = 6,400) |
| Pre-train data | `pretrain_t2t_mini.jsonl` (~1.2 GB) | **`pretrain_t2t.jsonl`** (~10 GB) |
| SFT data | `sft_t2t_mini.jsonl` (~1.6 GB) | **`sft_t2t.jsonl`** (~14 GB) |
| Model | `hidden_size=256`, 2 layers | **`hidden_size=768`, 8 layers** (~64 M params, MiniMind-3 main branch) |
| Trainer | inline demo loop | The production scripts `trainer/train_pretrain.py` and `trainer/train_full_sft.py` |
| Output | toy `.pth` checkpoints | `out/pretrain_768.pth`, `out/full_sft_768.pth` |

> **Hardware note.** A full pre-training pass on the 10 GB `pretrain_t2t.jsonl`
> takes roughly **2 hours on a single RTX 3090** per upstream README. On a free
> Colab **T4** expect 6–10× longer. Run on **A100 / L4** (Colab Pro) if possible,
> or reduce `EPOCHS` / use a subset for a sub-full reproduction.


## Step 0 · Environment check

Make sure a GPU is attached: **Runtime → Change runtime type → GPU**.
Recommended: A100 / L4 / V100. T4 works but is slow.

In [ ]:
!nvidia-smi || echo "No GPU detected — change the runtime type to GPU before continuing."
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Total memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Step 1 · (Optional) Mount Google Drive

We mount Drive so we can:

1. **Cache the datasets** under `/content/drive/MyDrive/minimind_data` — re-running
   the notebook then skips the multi-GB re-download.
2. **Save checkpoints** to Drive as a fallback if `files.download()` of the final
   `.pth` files fails (Colab's `files.download` can flake on >100 MB files).

If you don't want to use Drive, set `USE_DRIVE = False` and the notebook will
still work — you just lose the cache / fallback save.


In [ ]:
USE_DRIVE = True  # set to False to skip Google Drive entirely

DRIVE_ROOT = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = '/content/drive/MyDrive/minimind_serious'
        import os
        os.makedirs(DRIVE_ROOT, exist_ok=True)
        os.makedirs(f'{DRIVE_ROOT}/dataset', exist_ok=True)
        os.makedirs(f'{DRIVE_ROOT}/out', exist_ok=True)
        print("Drive mounted. Files will be cached under:", DRIVE_ROOT)
    except Exception as e:
        print("Could not mount Drive (running outside Colab?):", e)
        USE_DRIVE = False
        DRIVE_ROOT = None


## Step 2 · Clone the repo and install dependencies

The production training scripts (`trainer/train_pretrain.py`,
`trainer/train_full_sft.py`) and the real tokenizer files
(`model/tokenizer.json`, `model/tokenizer_config.json`) live in the
`Boyu-Zhang-UOI/minimind-colab` repository. We clone it so we can run the
trainers directly instead of duplicating their code in the notebook.

In [ ]:
%cd /content
import os
if not os.path.isdir('/content/minimind-colab'):
    !git clone --depth 1 https://github.com/Boyu-Zhang-UOI/minimind-colab.git
%cd /content/minimind-colab
!ls


In [ ]:
# Install training dependencies. We use loose lower-bound pins to ensure the
# tokenizer / chat-template / hf_transfer features we rely on are available;
# the repo's requirements.txt also works but pins exact versions intended for
# local training, which can be heavier than what Colab needs.
!pip install -q -U \
    "transformers>=4.40,<5" \
    "datasets>=2.18" \
    "huggingface_hub>=0.23" \
    "hf_transfer" \
    "tokenizers>=0.15" \
    "accelerate>=0.30" \
    "sentencepiece"

# hf_transfer dramatically speeds up multi-GB downloads from the HF Hub.
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


## Step 3 · Download the **full** datasets

We pull the two full-size files from `jingyaogong/minimind_dataset`:

* `pretrain_t2t.jsonl` (~10 GB) — pre-training corpus
* `sft_t2t.jsonl` (~14 GB) — supervised fine-tuning corpus

Total ≈ **24 GB**, which fits within Colab's standard `/content` disk
(≈ 100 GB on most runtimes). If Drive is mounted, the files are placed under
`Drive/minimind_serious/dataset` and symlinked into `dataset/` so subsequent
runs skip the download.

> If you only have a few hours of GPU budget, set `USE_MINI = True` to use
> the smaller `*_mini` files instead — the rest of the notebook is identical.


In [ ]:
USE_MINI = False  # set True for a quick run on the 1–1.6 GB *_mini files

from huggingface_hub import hf_hub_download
import os, shutil

REPO_ID = "jingyaogong/minimind_dataset"

if USE_MINI:
    PRETRAIN_FILE = "pretrain_t2t_mini.jsonl"
    SFT_FILE      = "sft_t2t_mini.jsonl"
else:
    PRETRAIN_FILE = "pretrain_t2t.jsonl"
    SFT_FILE      = "sft_t2t.jsonl"

# Where to ultimately put the files (the trainer scripts default to ../dataset/...)
LOCAL_DATASET_DIR = "/content/minimind-colab/dataset"
os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)

# Drive cache (if available) so we don't re-download on every session.
CACHE_DIR = f"{DRIVE_ROOT}/dataset" if DRIVE_ROOT else LOCAL_DATASET_DIR

def fetch(name):
    cache_path = os.path.join(CACHE_DIR, name)
    local_path = os.path.join(LOCAL_DATASET_DIR, name)
    if os.path.exists(cache_path):
        print(f"[cache] using cached {name} from {cache_path}")
    else:
        print(f"[download] {name} → {cache_path}")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=name,
            repo_type="dataset",
            local_dir=CACHE_DIR,
        )
    # Make sure the file is reachable at the path the trainer expects.
    if cache_path != local_path:
        if os.path.islink(local_path) or os.path.isfile(local_path):
            os.unlink(local_path)
        elif os.path.isdir(local_path):
            shutil.rmtree(local_path)
        os.symlink(cache_path, local_path)
    size_gb = os.path.getsize(local_path) / 1024**3
    print(f"[ok] {name}: {size_gb:.2f} GB at {local_path}")

fetch(PRETRAIN_FILE)
fetch(SFT_FILE)


In [ ]:
# Quick sanity: count lines and inspect one example from each.
import json

def head(path):
    with open(path, 'r', encoding='utf-8') as f:
        first = f.readline()
    return json.loads(first)

pre = f"/content/minimind-colab/dataset/{PRETRAIN_FILE}"
sft = f"/content/minimind-colab/dataset/{SFT_FILE}"

print("Pre-training sample:")
print(json.dumps(head(pre), ensure_ascii=False)[:400], "...\n")
print("SFT sample:")
print(json.dumps(head(sft), ensure_ascii=False)[:400], "...\n")


## Step 4 · Verify the **real** MiniMind tokenizer

The repo ships the production BPE tokenizer in `model/tokenizer.json` /
`model/tokenizer_config.json`. The trainer scripts auto-load it via
`AutoTokenizer.from_pretrained('../model')` inside `trainer_utils.init_model`.
Let's confirm it loads and that the chat template (used by `SFTDataset`)
behaves correctly.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('/content/minimind-colab/model')
print("Vocab size:", tok.vocab_size)
print("BOS / EOS / PAD:", tok.bos_token_id, tok.eos_token_id, tok.pad_token_id)

sample = tok("你好，MiniMind！Hello world.", add_special_tokens=False)
print("Sample ids:", sample.input_ids[:30])
print("Round trip:", tok.decode(sample.input_ids))

# Chat template — used by SFTDataset to mark assistant spans.
chat = [
    {"role": "user",      "content": "What is 2 + 2?"},
    {"role": "assistant", "content": "4."},
]
print("\nChat template render:\n", tok.apply_chat_template(chat, tokenize=False))


## Step 5 · Phase 1 — Pre-training

We invoke `trainer/train_pretrain.py` directly. This is the *exact* script the
upstream repo uses, so any results we get are an apples-to-apples reproduction.

Key arguments (defaults match the upstream MiniMind-3 main branch):

| Flag | Value | Meaning |
|---|---|---|
| `--hidden_size 768 --num_hidden_layers 8` | ~64 M-param config | MiniMind-3 main branch |
| `--max_seq_len 340` | upstream default for pre-train | balance speed vs. context |
| `--epochs 1` | one full pass over `pretrain_t2t.jsonl` | upstream uses 1–2 |
| `--batch_size 32 --accumulation_steps 8` | effective batch 256 | T4-friendly; raise on A100/L4 |
| `--dtype bfloat16` | mixed precision | use `float16` if `bfloat16` is unsupported |
| `--save_dir ./out` | where the final `.pth` is written | becomes `out/pretrain_768.pth` |

Output checkpoint: **`out/pretrain_768.pth`**.


In [ ]:
# Tune to your runtime. On a T4 you may want batch_size=16 and accumulation_steps=16.
PRETRAIN_EPOCHS = 1
PRETRAIN_BATCH  = 32
PRETRAIN_ACCUM  = 8
PRETRAIN_DTYPE  = "bfloat16"   # use "float16" if bfloat16 isn't supported on your GPU

# Detect bf16 support and downgrade automatically.
import torch
if PRETRAIN_DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    print("bfloat16 not supported on this GPU — falling back to float16.")
    PRETRAIN_DTYPE = "float16"

print("Will run pretraining with dtype =", PRETRAIN_DTYPE)


In [ ]:
%cd /content/minimind-colab/trainer
!python train_pretrain.py \
    --save_dir ../out \
    --save_weight pretrain \
    --epochs {PRETRAIN_EPOCHS} \
    --batch_size {PRETRAIN_BATCH} \
    --accumulation_steps {PRETRAIN_ACCUM} \
    --dtype {PRETRAIN_DTYPE} \
    --hidden_size 768 \
    --num_hidden_layers 8 \
    --max_seq_len 340 \
    --learning_rate 5e-4 \
    --log_interval 100 \
    --save_interval 1000 \
    --num_workers 2 \
    --data_path ../dataset/{PRETRAIN_FILE}


In [ ]:
# Sanity check — the pretrain checkpoint should now exist.
import os
PRETRAIN_CKPT = "/content/minimind-colab/out/pretrain_768.pth"
assert os.path.exists(PRETRAIN_CKPT), f"Missing {PRETRAIN_CKPT}"
print(f"Pretrain checkpoint: {PRETRAIN_CKPT} ({os.path.getsize(PRETRAIN_CKPT)/1024**2:.1f} MB)")

# Mirror to Drive immediately so a Colab disconnect doesn't lose the result.
if DRIVE_ROOT:
    import shutil
    dst = f"{DRIVE_ROOT}/out/pretrain_768.pth"
    shutil.copy2(PRETRAIN_CKPT, dst)
    print("Backed up to", dst)


## Step 6 · Phase 2 — Supervised Fine-Tuning (SFT)

`trainer/train_full_sft.py` loads the pre-trained weights via
`--from_weight pretrain` (which expands to `out/pretrain_768.pth`) and continues
training on the chat-format `sft_t2t.jsonl`. The `SFTDataset` masks user / system
turns to `-100` so loss is only computed on assistant replies (see
`dataset/lm_dataset.py`).

Output checkpoint: **`out/full_sft_768.pth`**.


In [ ]:
SFT_EPOCHS = 2
SFT_BATCH  = 16
SFT_ACCUM  = 1
SFT_DTYPE  = PRETRAIN_DTYPE   # same dtype decision as pretraining


In [ ]:
%cd /content/minimind-colab/trainer
!python train_full_sft.py \
    --save_dir ../out \
    --save_weight full_sft \
    --from_weight pretrain \
    --epochs {SFT_EPOCHS} \
    --batch_size {SFT_BATCH} \
    --accumulation_steps {SFT_ACCUM} \
    --dtype {SFT_DTYPE} \
    --hidden_size 768 \
    --num_hidden_layers 8 \
    --max_seq_len 768 \
    --learning_rate 1e-5 \
    --log_interval 100 \
    --save_interval 1000 \
    --num_workers 2 \
    --data_path ../dataset/{SFT_FILE}


In [ ]:
import os
SFT_CKPT = "/content/minimind-colab/out/full_sft_768.pth"
assert os.path.exists(SFT_CKPT), f"Missing {SFT_CKPT}"
print(f"SFT checkpoint: {SFT_CKPT} ({os.path.getsize(SFT_CKPT)/1024**2:.1f} MB)")

if DRIVE_ROOT:
    import shutil
    dst = f"{DRIVE_ROOT}/out/full_sft_768.pth"
    shutil.copy2(SFT_CKPT, dst)
    print("Backed up to", dst)


## Step 7 · Sanity-check generation

Quickly load the SFT model and generate a couple of replies to confirm the
training run is sensible. We use the model's `generate` method (same code
path as `eval_llm.py`).

In [ ]:
import sys, os, torch
sys.path.append('/content/minimind-colab')

from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

cfg = MiniMindConfig(hidden_size=768, num_hidden_layers=8, use_moe=False)
tok = AutoTokenizer.from_pretrained('/content/minimind-colab/model')
model = MiniMindForCausalLM(cfg)
state = torch.load('/content/minimind-colab/out/full_sft_768.pth', map_location='cpu')
model.load_state_dict(state, strict=False)
model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')

def chat(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id,
        )
    return tok.decode(out[0, ids.size(1):], skip_special_tokens=True)

for q in ["Please introduce yourself in one sentence.",
          "用一句话介绍一下你自己。",
          "What is the capital of France?"]:
    print(f"USER:      {q}")
    print(f"ASSISTANT: {chat(q)}\n")


## Step 8 · Download the trained checkpoints

We attempt a direct browser download of `pretrain_768.pth` and `full_sft_768.pth`.
`google.colab.files.download` can fail / hang for files larger than ~100 MB,
so when that happens we fall back to **copying the files into Google Drive**
(which we already mirrored after each phase as a safety net in steps 5 & 6).


In [ ]:
import os, shutil

CHECKPOINTS = [
    "/content/minimind-colab/out/pretrain_768.pth",
    "/content/minimind-colab/out/full_sft_768.pth",
]

def try_browser_download(path):
    try:
        from google.colab import files
        print(f"Attempting browser download: {path} ({os.path.getsize(path)/1024**2:.1f} MB)")
        files.download(path)
        return True
    except Exception as e:
        print(f"  browser download failed: {e}")
        return False

def save_to_drive(path):
    if not DRIVE_ROOT:
        print(f"  (no Drive mounted — file remains at {path}; re-mount Drive and re-run this cell to back it up)")
        return False
    dst = f"{DRIVE_ROOT}/out/{os.path.basename(path)}"
    shutil.copy2(path, dst)
    print(f"  saved to Drive: {dst}")
    return True

for ckpt in CHECKPOINTS:
    if not os.path.exists(ckpt):
        print(f"[skip] missing {ckpt}")
        continue
    print(f"\n=== {os.path.basename(ckpt)} ===")
    if not try_browser_download(ckpt):
        save_to_drive(ckpt)

print("\nDone. Final artifacts:")
for ckpt in CHECKPOINTS:
    if os.path.exists(ckpt):
        print(f"  local:  {ckpt}  ({os.path.getsize(ckpt)/1024**2:.1f} MB)")
    if DRIVE_ROOT:
        d = f"{DRIVE_ROOT}/out/{os.path.basename(ckpt)}"
        if os.path.exists(d):
            print(f"  drive:  {d}")


## Notes on reproducing the upstream results

* Upstream MiniMind-3 reports the main branch is trained with
  `hidden_size=768`, 8 transformer layers, vocab 6,400 — total ≈ 64 M parameters.
* Pre-training: 1–2 epochs over `pretrain_t2t.jsonl` (~10 GB).
* SFT: 2 epochs over `sft_t2t.jsonl` (~14 GB), starting from the pre-trained checkpoint.
* On a single RTX 3090 the upstream README quotes ~2 hours total. Free Colab T4 will be substantially slower; an A100 / L4 runtime is highly recommended for serious reproduction.
* If you hit OOM, lower `--batch_size` (e.g. 16 or 8) and raise `--accumulation_steps` so the effective batch size is preserved.
* If a session disconnects mid-training, re-run the same trainer command with `--from_resume 1` to pick up from the last `lm_checkpoint` snapshot in `checkpoints/`.
